# Markov-Kette: Zufallsbewegung auf einem Ring

Dieses Notebook untersucht eine einfache zeitdiskrete Markov-Kette. Ein Teilchen bewegt sich zufällig zwischen Gitterplätzen; seine nächste Position hängt nur von seiner aktuellen Position ab und nicht von der bisherigen Bewegung.

In [ ]:
# Import all the numerics and plotting packages
import numpy as np
import scipy as scipy
import matplotlib
from matplotlib import pyplot as plt
from IPython.core.pylabtools import figsize, getfigs
import matplotlib.animation as animation
from IPython.display import Video
from matplotlib.animation import FuncAnimation
import time
from scipy.linalg import eigvals

## Modell und Übergangsregel

Wir betrachten $N$ Gitterplätze $0,\ldots,N-1$ mit periodischen Randbedingungen, also $N\equiv0$. In jedem Zeitschritt springt das Teilchen mit Wahrscheinlichkeit $p$ nach rechts, mit Wahrscheinlichkeit $p$ nach links und bleibt mit Wahrscheinlichkeit $1-2p$ am selben Ort.

Die Zufallsentscheidung wird mit einer gleichverteilten Zahl $r\in[0,1)$ getroffen: Für $r<p$ erfolgt ein Schritt nach rechts, für $p\le r<2p$ ein Schritt nach links. Andernfalls bleibt die Position unverändert. Die Modulo-Operation setzt die periodischen Randbedingungen um.

In [ ]:
p=0.1
np.random.seed() # seed for random function 

def step(xn,p):   
    rand=np.random.uniform(0,1)
    if rand<p:
        #jump to the right
        xn1 = (xn+1)%N     # %N is due to the boundary conditions
    elif rand <2*p:
        #jump to the left
        xn1 = (xn-1)%N
        if xn1 == -1:
            xn1 = N-1
    else:
        xn1=xn
    return xn1


def random_walk(nmax,N):   
    j=N//2
    for n in range(nmax):
        rand=np.random.uniform(0,1)
        if rand<p:
            #jump to the right
            j = (j+1)%N     # %N is due to the boundary conditions
        elif rand <2*p:
            #jump to the left
            j = (j-1)%N
            #for the case j=0 -> j=N-1
            if j == -1:
                j = N-1
    return j


## Einzelne Trajektorien

Die folgenden Kurven zeigen verschiedene Realisierungen desselben Zufallsprozesses. Obwohl alle Läufe am mittleren Gitterplatz beginnen und dieselben Übergangswahrscheinlichkeiten besitzen, unterscheiden sich ihre konkreten Wege durch die Zufallszahlen.

In [ ]:
# plot few trajectories: of length nmax
N=40 
p=0.1
nmax=100

def get_trajectory(x_initial,nmax,p):
    traj=np.zeros(nmax+1)
    traj[0]=x_initial
    for m in range(nmax):
        traj[m+1]=step(traj[m],p)
    return traj
    
    
fig , ax = plt.subplots()
plt.ylim(0,N-1)
ax.plot(get_trajectory(N//2,nmax,p),'.-')
ax.plot(get_trajectory(N//2,nmax,p),'.-')
ax.plot(get_trajectory(N//2,nmax,p),'.-')
ax.plot(get_trajectory(N//2,nmax,p),'.-')
ax.plot(get_trajectory(N//2,nmax,p),'.-')
ax.plot(get_trajectory(N//2,nmax,p),'.-')
ax.plot(get_trajectory(N//2,nmax,p),'.-')
plt.show()



## Verteilung nach einer festen Schrittzahl

Viele unabhängige Läufe liefern ein normiertes Histogramm der Endpositionen. Dieses Monte-Carlo-Ergebnis wird mit der exakten Verteilung verglichen. Dazu wirkt die Übergangsmatrix $W$ wiederholt auf den Anfangszustand: $\boldsymbol{P}^{(n)}=W^n\boldsymbol{P}^{(0)}$.

In [ ]:
nmax=10
mmax=10000 # histogram points 
P = np.zeros(N)
for s in range(mmax):
    j=random_walk(nmax,N)
    P[j]+=1
    

#For comparison we generate also the exact distribution, by applying the transition matrix
pex = np.zeros(N)
pex[N//2] = 1.0     # initialization of pstat
W = np.zeros((N,N))
for j in range(N):    # initialization of W
    W[j][j] = 1. - 2. * p
    j1 = (j+1)%N
    W[j,j1]=p
    W[j1,j]=p
for n in range(nmax):
    pex = np.matmul(W,pex)
    
print(W)
# Here we plot the results
N_array = np.arange(N)
norm_P = P/mmax

fig,ax=plt.subplots(figsize=(4, 6))

ax.bar(N_array,norm_P, label = "Random walk after "+str(n)+" steps",width=1,color="skyblue", edgecolor="black")
ax.plot(N_array,pex,color ='b',label='Exact distribution')
ax.legend()
plt.show()

## Zeitentwicklung und stationäre Verteilung

Für verschiedene Schrittzahlen wird sichtbar, wie sich die anfangs lokalisierte Verteilung über den Ring ausbreitet. Nach vielen Schritten nähert sie sich der Gleichverteilung $P(c)=1/N$, der stationären Verteilung dieser symmetrischen Markov-Kette.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Parameters
N = 20        # Total number of points in the distribution
mmax = 10000  # Number of histogram samples
p = 0.1       # Probability for each step in the random walk


# Values of nmax for each plot
nmax_values = [0, 1, 2, 10, 1000]

# Set up the 1x6 grid of plots with shared y-axis
fig, axs = plt.subplots(1, len(nmax_values), figsize=(12, 6), sharey=True)
fig.suptitle("Random Walk Distribution for Different Steps (nmax)", y=1.05)

# Iterate over each nmax value, compute the histogram, and plot
for idx, nmax in enumerate(nmax_values):
    # Reset the histogram for each nmax value
    P = np.zeros(N)
    
    # Run the random walk mmax times and build the histogram
    for _ in range(mmax):
        j = random_walk(nmax, N)
        P[j] += 1
    
    # Normalize the histogram
    norm_P = P / mmax

    # Compute the exact distribution for comparison
    pex = np.zeros(N)
    pex[N // 2] = 1.0  # Initialize exact distribution at center
    W = np.zeros((N, N))
    for j in range(N):  # Initialize transition matrix W
        W[j][j] = 1. - 2. * p
        j1 = (j + 1) % N
        W[j, j1] = p
        W[j1, j] = p
    for _ in range(nmax):
        pex = np.matmul(W, pex)

    # Plot the histogram and exact distribution on the current subplot
    N_array = np.arange(N)
    axs[idx].bar(N_array, norm_P, width=1, color="skyblue", edgecolor="black")
    axs[idx].plot(N_array, pex, color='b')
    axs[idx].set_title(f"n = {nmax}")
    axs[idx].set_xlabel("c")
    
# Set the y-axis label on the first subplot only
axs[0].set_ylabel("$P^{(n)}(c)$")

# Adjust layout and show the plot
plt.tight_layout()
plt.subplots_adjust(top=0.8)  # Adjust top to fit the main title
plt.show()
